In [1]:
import ast
import json
import time
from pathlib import Path
import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm
from google.colab import drive
from google.colab import userdata

In [2]:
drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/thesis_results/SCOTBESS_labeling/v2")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROJECT_DIR / "SCOTBESS_CLEAN.csv"
DEFINITIONS_PATH = (PROJECT_DIR / "SCOTBESS_labels_definitions_v2.xlsx")

PREDICTIONS_PATH = (PROJECT_DIR / "SCOTBESS_full_terra_low_predictions.csv")
FINAL_DATA_PATH = (PROJECT_DIR / "SCOTBESS_FULL_ANNOTATED_TERRA_LOW.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
#labels
labels_df = pd.read_excel(DEFINITIONS_PATH)

required_definition_columns = {"label", "definition"}
missing = required_definition_columns - set(labels_df.columns)

if missing:
    raise ValueError(f"Definitions file is missing columns: {sorted(missing)}")

labels_df["label"] = labels_df["label"].astype(str).str.strip()
labels_df["definition"] = labels_df["definition"].astype(str).str.strip()

if labels_df["label"].duplicated().any():
    raise ValueError("Duplicate labels found in definitions file.")

label_descriptions = dict(zip(labels_df["label"], labels_df["definition"]))

LABELS = list(label_descriptions)
LABEL_SET = set(LABELS)
LABEL_ORDER = {label: position for position, label in enumerate(LABELS)}

print(f"Labels loaded: {len(LABELS)}")
display(labels_df)

Labels loaded: 20


,label,definition
0,Wildlife and Ecology,"Impacts on wildlife, protected species, habita..."
1,Traffic,"Vehicle movements, HGV routing, road capacity ..."
2,Fire and Explosion Risk,"The likelihood, severity, or spread of battery..."
3,"Landscape, Visual and Heritage Impact","Visual intrusion, landscape and rural characte..."
4,"Consultation, Transparency and Information","Public engagement, notification, access to inf..."
5,Emergency Planning and Response,"Emergency response plans, evacuation, fire-ser..."
6,Water and Soil Contamination,"Hazardous substances, chemical leakage or poll..."
7,Cumulative Impact,Concerns that the proposal would add to the co...
8,Light Pollution,Adverse effects of operational or security lig...
9,Noise,"Noise generated by the development, including ..."


In [14]:
#dataset
df = pd.read_csv(DATA_PATH)

required_columns = {"document_id", "project", "source", "final_masked_text"}
missing = required_columns - set(df.columns)

if missing:
    raise ValueError(f"Dataset is missing columns: {sorted(missing)}")

if df["document_id"].duplicated().any():
    raise ValueError("document_id must be unique.")

if df["final_masked_text"].isna().any():
    raise ValueError("final_masked_text contains missing values.")

df["final_masked_text"] = df["final_masked_text"].astype(str).str.strip()

if df["final_masked_text"].eq("").any():
    raise ValueError("final_masked_text contains blank responses.")

print(f"Responses ready for annotation: {len(df):,}")
display(df[["document_id", "project", "source", "final_masked_text"]].head())

Responses ready for annotation: 1,675


,document_id,project,source,final_masked_text
0,0,Aberdeen_City_210665_DPP,comment_structure,The ground on which the proposed energy site i...
1,1,Aberdeen_City_210665_DPP,no_structure,We have serious reservations that there has no...
2,2,Aberdeen_City_220026_DPP,no_structure,[ORGANIZATION] opposes the application on the ...
3,3,Aberdeen_City_231134_DPP,comment_structure,I have read with interest the summary document...
4,4,Aberdeen_City_231134_DPP,comment_structure,I note from a review of the Planning Statement...


In [7]:
FINAL_CONFIG = {
    "config": "terra_low",
    "model": "gpt-5.6-terra",
    "effort": "low"}

MAX_RETRIES = 3

In [8]:
api_key = userdata.get("open")
client = OpenAI(api_key=api_key)

In [10]:
#same definitions and schema as used when choosing the right model

definitions_block = "\n".join(
    f"{i + 1}. {row.label}: {row.definition}"
    for i, row in labels_df.iterrows())

ANNOTATION_INSTRUCTIONS = f"""You are an expert in annotating Scottish planning representations concerning battery energy storage system developments.

Assign every label supported by the response. This is a multi-label classification task. If none of the labels is supported, return an empty labels list.

Use the provided label definitions as the sole basis for assigning labels. Follow their inclusion criteria, exclusions, and boundaries carefully.

Rules:
- Use only exact label names from the permitted label set.
- Do not infer concerns, opinions, causes, or consequences that are not expressed.
- A single sentence or clause is sufficient when it clearly supports a label.
- Do not assign a label solely because a related topic or keyword is mentioned incidentally.
- Return only the JSON object, with no explanation or extra text.

PERMITTED LABELS AND DEFINITIONS
{definitions_block}
"""

LABEL_SCHEMA = {
    "type": "object",
    "properties": {
        "labels": {
            "type": "array",
            "description": "All and only the substantively supported labels.",
            "items": {"type": "string", "enum": LABELS},}},
    "required": ["labels"],
    "additionalProperties": False}



In [11]:
def parse_label_list(value) -> list[str]:
    if isinstance(value, list):
        values = value
    elif pd.isna(value) or not str(value).strip():
        raise ValueError("Blank label value.")
    else:
        text = str(value).strip()
        try:
            values = json.loads(text)
        except json.JSONDecodeError:
            try:
                values = ast.literal_eval(text)
            except (ValueError, SyntaxError) as exc:
                raise ValueError(f"Could not parse labels: {value!r}") from exc

    if not isinstance(values, list):
        raise ValueError(f"Expected a list, received: {value!r}")

    if not all(isinstance(label, str) for label in values):
        raise ValueError(f"Every label must be a string: {value!r}")

    values = [label.strip() for label in values]

    unknown = sorted(set(values) - LABEL_SET)
    if unknown:
        raise ValueError(f"Unknown labels: {unknown}")

    return sorted(set(values), key=LABEL_ORDER.get)

In [20]:
def classify_one(row: dict) -> dict:
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.responses.create(
                model=FINAL_CONFIG["model"],
                reasoning={"effort": FINAL_CONFIG["effort"]},
                instructions=ANNOTATION_INSTRUCTIONS,
                input=f"Classify the following consultation response.\n\nRESPONSE:\n{row['final_masked_text']}",
                text={"format": {
                    "type": "json_schema",
                    "name": "scotbess_labels",
                    "strict": True,
                    "schema": LABEL_SCHEMA}},
                max_output_tokens=500, #for one response the output token limit was increased to 1000, as it was failing
                store=False)
            #recording the reason if the response is incomplete
            if response.status != "completed":
                reason = response.incomplete_details.reason if response.incomplete_details else "unknown"
                raise RuntimeError(f"Response status: {response.status}, reason: {reason}")

            parsed = json.loads(response.output_text)
            labels = parse_label_list(parsed["labels"])

            return {
                "document_id": row["document_id"],
                "config": FINAL_CONFIG["config"],
                "model": FINAL_CONFIG["model"],
                "reasoning_effort": FINAL_CONFIG["effort"],
                "predicted_labels": json.dumps(labels, ensure_ascii=False),
                "error": ""}

        except Exception as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                time.sleep(2 ** (attempt - 1))

    return {
        "document_id": row["document_id"],
        "config": FINAL_CONFIG["config"],
        "model": FINAL_CONFIG["model"],
        "reasoning_effort": FINAL_CONFIG["effort"],
        "predicted_labels": "",
        "error": repr(last_error)}

In [15]:
#test to see wheter it works
test_rows = df.head(3).to_dict("records")

test_results = [classify_one(row) for row in tqdm(test_rows)]
test_results = pd.DataFrame(test_results)

display(test_results[["document_id", "predicted_labels", "error"]])

  0%|          | 0/3 [00:00<?, ?it/s]

,document_id,predicted_labels,error
0,0,"[""Site Selection""]",
1,1,"[""Fire and Explosion Risk"", ""Site Selection"", ...",
2,2,"[""Fire and Explosion Risk"", ""Emergency Plannin...",


In [21]:
#FULL ANNOTATION

if PREDICTIONS_PATH.exists():
    predictions = pd.read_csv(PREDICTIONS_PATH)
else:
    predictions = pd.DataFrame()

completed = set()

if not predictions.empty:
    successful = predictions[predictions["error"].fillna("").eq("")]
    completed = set(successful["document_id"])

jobs = [row for row in df.to_dict("records") if row["document_id"] not in completed]

print(f"Already completed: {len(completed)}")
print(f"API calls remaining: {len(jobs)}")

for row in tqdm(jobs):
    result = classify_one(row)

    predictions = pd.concat([predictions, pd.DataFrame([result])], ignore_index=True)
    predictions = predictions.drop_duplicates(subset=["document_id"], keep="last").sort_values("document_id")
    predictions.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

errors = predictions[~predictions["error"].fillna("").eq("")]

print(f"Saved: {PREDICTIONS_PATH}")
print(f"Failed calls: {len(errors)}")

if len(errors):
    display(errors[["document_id", "error"]])

Already completed: 1674
API calls remaining: 1


  0%|          | 0/1 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/thesis_results/SCOTBESS_labeling/v2/SCOTBESS_full_terra_low_predictions.csv
Failed calls: 0


In [22]:
successful = predictions[predictions["error"].fillna("").eq("")].copy()
errors = predictions[~predictions["error"].fillna("").eq("")].copy()

print(f"Dataset rows: {len(df)}")
print(f"Successful annotations: {len(successful)}")
print(f"Failed annotations: {len(errors)}")

if len(successful) != len(df):
    raise ValueError(f"Annotation incomplete: {len(successful)}/{len(df)} successful.")

if set(successful["document_id"]) != set(df["document_id"]):
    raise ValueError("Prediction document IDs do not match the source dataset.")

print("Full annotation completed successfully.")

Dataset rows: 1675
Successful annotations: 1675
Failed annotations: 0
Full annotation completed successfully.


In [23]:
successful["labels"] = successful["predicted_labels"].apply(parse_label_list)
successful["labels"] = successful["labels"].apply(lambda x: json.dumps(x, ensure_ascii=False))

In [24]:
annotated_df = df.merge(successful[["document_id", "labels"]], on="document_id", how="left", validate="one_to_one")

if annotated_df["labels"].isna().any():
    raise ValueError("Some responses are missing labels after the merge.")

print(f"Annotated dataset rows: {len(annotated_df)}")
display(annotated_df[["document_id", "final_masked_text", "labels"]].head())

Annotated dataset rows: 1675


,document_id,final_masked_text,labels
0,0,The ground on which the proposed energy site i...,"[""Site Selection""]"
1,1,We have serious reservations that there has no...,"[""Fire and Explosion Risk"", ""Consultation, Tra..."
2,2,[ORGANIZATION] opposes the application on the ...,"[""Fire and Explosion Risk"", ""Emergency Plannin..."
3,3,I have read with interest the summary document...,"[""Consultation, Transparency and Information""]"
4,4,I note from a review of the Planning Statement...,"[""Grid Connection and Electrical Infrastructure""]"


In [26]:
annotated_df.to_csv(FINAL_DATA_PATH, index=False, encoding="utf-8")
print(f"Saved: {FINAL_DATA_PATH}")

Saved: /content/drive/MyDrive/thesis_results/SCOTBESS_labeling/v2/SCOTBESS_FULL_ANNOTATED_TERRA_LOW.csv


In [27]:
#quality check

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

VALIDATION_SET_PATH = PROJECT_DIR / "SCOTBESS_annotation_validation_20.xlsx"
VALIDATION_PREDICTIONS_PATH = PROJECT_DIR / "SCOTBESS_annotation_validation_terra_low.csv"

validation_set = pd.read_excel(VALIDATION_SET_PATH)
validation_predictions = pd.read_csv(VALIDATION_PREDICTIONS_PATH)
full_annotated = pd.read_csv(FINAL_DATA_PATH)

print(f"Manual validation cases: {len(validation_set)}")
print(f"Original validation predictions: {len(validation_predictions)}")
print(f"Full annotated dataset: {len(full_annotated)}")

Manual validation cases: 20
Original validation predictions: 20
Full annotated dataset: 1675


In [28]:
validation_set["manual_labels"] = validation_set["manual_gold_labels"].apply(parse_label_list)

validation_predictions = validation_predictions[validation_predictions["error"].fillna("").eq("")].copy()
validation_predictions["validation_terra_labels"] = (validation_predictions["predicted_labels"].apply(parse_label_list))

full_annotated["full_run_terra_labels"] = full_annotated["labels"].apply(parse_label_list)

In [29]:
qc = validation_set[
    ["pilot_id", "document_id", "final_masked_text", "manual_labels"]].merge(
    validation_predictions[["document_id", "validation_terra_labels"]],
    on="document_id",
    how="left",
    validate="one_to_one").merge(
    full_annotated[["document_id", "full_run_terra_labels"]],
    on="document_id",
    how="left",
    validate="one_to_one")

if len(qc) != 20:
    raise ValueError(f"Expected 20 validation cases, found {len(qc)}.")

if qc[["validation_terra_labels", "full_run_terra_labels"]].isna().any().any():
    raise ValueError("Some validation cases are missing Terra predictions.")

print(f"Quality-check cases: {len(qc)}")

Quality-check cases: 20


In [30]:
mlb = MultiLabelBinarizer(classes=LABELS)
mlb.fit([LABELS])

def compare_labels(reference, prediction, name):
    y_true = mlb.transform(reference)
    y_pred = mlb.transform(prediction)

    return {
        "comparison": name,
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "sample_f1": f1_score(y_true, y_pred, average="samples", zero_division=0),
        "exact_match": accuracy_score(y_true, y_pred)}

qc_metrics = pd.DataFrame([
    compare_labels(
        qc["manual_labels"],
        qc["validation_terra_labels"],
        "Manual vs original Terra validation"),
    compare_labels(
        qc["manual_labels"],
        qc["full_run_terra_labels"],
        "Manual vs full-run Terra"),
    compare_labels(
        qc["validation_terra_labels"],
        qc["full_run_terra_labels"],
        "Original Terra vs full-run Terra")])

display(qc_metrics.style.format({
    "micro_precision": "{:.3f}",
    "micro_recall": "{:.3f}",
    "micro_f1": "{:.3f}",
    "sample_f1": "{:.3f}",
    "exact_match": "{:.3f}"}))

,comparison,micro_precision,micro_recall,micro_f1,sample_f1,exact_match
0,Manual vs original Terra validation,0.963,0.977,0.970,0.979,0.700
1,Manual vs full-run Terra,0.956,0.985,0.970,0.956,0.650
2,Original Terra vs full-run Terra,0.963,0.978,0.970,0.958,0.650


In [32]:
#inspecting where full-run Terra labels disagree with  manual annotation
qc_disagreements = qc[qc.apply(lambda row: set(row["manual_labels"]) != set(row["full_run_terra_labels"]), axis=1)].copy()

qc_disagreements["missing_labels"] = qc_disagreements.apply(lambda row: sorted(set(row["manual_labels"]) - set(row["full_run_terra_labels"])), axis=1)

qc_disagreements["extra_labels"] = qc_disagreements.apply(lambda row: sorted(set(row["full_run_terra_labels"]) - set(row["manual_labels"])), axis=1)

print(f"Disagreements: {len(qc_disagreements)}/{len(qc)}")

display(qc_disagreements[
    [
        "pilot_id",
        "manual_labels",
        "full_run_terra_labels",
        "missing_labels",
        "extra_labels",
        "final_masked_text"]])

Disagreements: 7/20


,pilot_id,manual_labels,full_run_terra_labels,missing_labels,extra_labels,final_masked_text
2,4,"[Wildlife and Ecology, Fire and Explosion Risk...","[Wildlife and Ecology, Fire and Explosion Risk...",[],[Emergency Planning and Response],I would like to object to the proposed plan to...
6,8,"[Traffic, Fire and Explosion Risk, Landscape, ...","[Traffic, Fire and Explosion Risk, Landscape, ...",[],[Grid Connection and Electrical Infrastructure],Please find below a note of my strong objectio...
9,14,"[Wildlife and Ecology, Traffic, Fire and Explo...","[Wildlife and Ecology, Traffic, Fire and Explo...",[],[Grid Connection and Electrical Infrastructure],The [PLACE] Battery Energy Storage System (BES...
14,21,"[Traffic, Fire and Explosion Risk, Emergency P...","[Traffic, Fire and Explosion Risk, Landscape, ...",[],"[Landscape, Visual and Heritage Impact, Projec...",[APPLICATION_CODE] What is the battery storage...
15,22,"[Consultation, Transparency and Information, P...","[Consultation, Transparency and Information, P...",[Project Need],[],I understand that while planning permission in...
18,27,"[Fire and Explosion Risk, Residential Proximit...","[Fire and Explosion Risk, Site Selection, Resi...",[],[Site Selection],I am concerned about this battery storage faci...
19,30,"[Landscape, Visual and Heritage Impact, Agricu...","[Landscape, Visual and Heritage Impact]",[Agricultural Land],[],After viewing the information i do not think t...


In [34]:
qc_metrics.to_excel(PROJECT_DIR / "SCOTBESS_annotation_qc_metrics.xlsx", index=False)
QC_DISAGREEMENTS_PATH = PROJECT_DIR / "SCOTBESS_annotation_qc_disagreements.xlsx"

qc_disagreements[["pilot_id","manual_labels","full_run_terra_labels","missing_labels","extra_labels","final_masked_text"]].to_excel(QC_DISAGREEMENTS_PATH, index=False)
print(f"Saved: {QC_DISAGREEMENTS_PATH}")

Saved: /content/drive/MyDrive/thesis_results/SCOTBESS_labeling/v2/SCOTBESS_annotation_qc_disagreements.xlsx
